In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import pennylane as qml
import matplotlib.pyplot as plt

In [3]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

print(len(train_dataset))
print(len(test_dataset))

60000
10000


In [4]:
train_dataset = torch.utils.data.Subset(train_dataset, range(2000))
test_dataset = torch.utils.data.Subset(test_dataset, range(500))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [5]:
class ClassicalCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(

            # 1 × 28 × 28
            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=5,
                stride=1,
                padding=2
            ),

            nn.BatchNorm2d(16),
            nn.ReLU(),

            # 16 × 28 × 28 -> 16 × 14 × 14
            nn.MaxPool2d(2),

            # 16 × 14 × 14
            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=5,
                stride=1,
                padding=2
            ),

            nn.BatchNorm2d(32),
            nn.ReLU(),

            # 32 × 14 × 14 -> 32 × 7 × 7
            nn.MaxPool2d(2)
        )

    def forward(self, x):

        x = self.conv(x)

        # 32 × 7 × 7
        x = torch.flatten(x, start_dim=1)

        # 1568 features
        return x
    
cnn = ClassicalCNN()

x = torch.randn(8, 1, 28, 28)

y = cnn(x)

print(y.shape)

torch.Size([8, 1568])


In [6]:
feature_reduction = nn.Linear(1568, 20)

# The quantum circuit

In [7]:
n_qubits = 5
n_layers = 3

In [8]:
import torch
import pennylane as qml

device = torch.device("cuda")

n_qubits = 5
n_layers = 3

# Quantum simulator runs on NVIDIA GPU
qdev = qml.device(
    "lightning.gpu",
    wires=n_qubits,
    shots=None
)


@qml.qnode(
    qdev,
    interface="torch",
    diff_method="adjoint"
)
def quantum_circuit(inputs, weights):

    # Encode 5 classical features into 5 qubits
    qml.AngleEmbedding(
        inputs,
        wires=range(n_qubits),
        rotation="X"
    )

    # Trainable quantum circuit
    qml.StronglyEntanglingLayers(
        weights,
        wires=range(n_qubits)
    )

    # One output per qubit
    return [
        qml.expval(qml.PauliY(i))
        for i in range(n_qubits)
    ]


weight_shapes = {
    "weights": (
        n_layers,
        n_qubits,
        3
    )
}

In [9]:
weights = torch.randn(
    n_layers,
    n_qubits,
    3
)

inputs = torch.randn(n_qubits)

print(
    qml.draw(quantum_circuit)(
        inputs,
        weights
    )
)

0: ─╭AngleEmbedding(M0)─╭StronglyEntanglingLayers(M1)─┤  <Y>
1: ─├AngleEmbedding(M0)─├StronglyEntanglingLayers(M1)─┤  <Y>
2: ─├AngleEmbedding(M0)─├StronglyEntanglingLayers(M1)─┤  <Y>
3: ─├AngleEmbedding(M0)─├StronglyEntanglingLayers(M1)─┤  <Y>
4: ─╰AngleEmbedding(M0)─╰StronglyEntanglingLayers(M1)─┤  <Y>

M0 = 
tensor([ 0.1696, -0.0425, -1.1832, -0.5526, -0.3333])
M1 = 
tensor([[[-1.2320,  1.0069, -1.5224],
         [ 0.7103,  0.2417, -0.3932],
         [ 0.4829, -1.7807,  1.7160],
         [-1.0326, -0.0994,  1.4471],
         [-0.9977,  0.8205, -0.9111]],

        [[-0.4036, -0.5976,  1.1144],
         [ 1.9700,  1.6821,  1.4180],
         [-0.1938,  1.9375,  1.2468],
         [-2.0102, -0.3929,  1.7248],
         [ 0.6105,  0.6401,  0.0739]],

        [[-0.2343, -0.1425,  0.2452],
         [ 0.5048,  0.1189,  2.8276],
         [-0.2837,  0.0530, -0.3428],
         [ 1.0736, -1.4382, -1.7119],
         [-1.0899, -0.7452, -0.1153]]])


In [10]:
weight_shapes = {
    "weights": (
        n_layers,
        n_qubits,
        3
    )
}

quantum_layer = qml.qnn.TorchLayer(
    quantum_circuit,
    weight_shapes
)

In [11]:
x = torch.randn(4, 5)

output = quantum_layer(x)

print(output.shape)

torch.Size([4, 5])


In [12]:
class ParallelQuantumLayer(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.quantum_layers = torch.nn.ModuleList([
            qml.qnn.TorchLayer(
                quantum_circuit,
                weight_shapes
            )
            for _ in range(4)
        ])

    def forward(self, x):

        outputs = []

        for i, layer in enumerate(self.quantum_layers):

            start = i * n_qubits
            end = start + n_qubits

            x_part = x[:, start:end]

            q_out = layer(x_part)

            outputs.append(q_out)

        return torch.cat(outputs, dim=1)

In [13]:
parallel_q = ParallelQuantumLayer()

x = torch.randn(8, 20)
y = parallel_q(x)

print(y.shape)

torch.Size([8, 20])


In [14]:
class HQNNParallel(nn.Module):
    def __init__(self):
        super().__init__()
        # Classical feature extraction
        self.cnn = ClassicalCNN()
        # 1568 -> 20
        self.fc1 = nn.Linear(
            1568,
            20
        )
        self.bn1 = nn.BatchNorm1d(20)
        # Quantum component
        self.quantum = ParallelQuantumLayer()
        # Final classification
        self.fc2 = nn.Linear(
            20,
            10
        )

    def forward(self, x):
        # ---------------------------------
        # Classical CNN
        # ---------------------------------
        x = self.cnn(x)
        # x: [batch, 1568]
        # ---------------------------------
        # Dimensionality reduction
        # ---------------------------------
        x = self.fc1(x)
        x = self.bn1(x)
        x = torch.relu(x)
        # x: [batch, 20]
        # ---------------------------------
        # Quantum layer
        # ---------------------------------
        x = self.quantum(x)
        # x: [batch, 20]
        # ---------------------------------
        # Classification
        # ---------------------------------
        x = self.fc2(x)
        # x: [batch, 10]
        return x

In [15]:
model = HQNNParallel()

print(model)

HQNNParallel(
  (cnn): ClassicalCNN(
    (conv): Sequential(
      (0): Conv2d(1, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
      (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
  )
  (fc1): Linear(in_features=1568, out_features=20, bias=True)
  (bn1): BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (quantum): ParallelQuantumLayer(
    (quantum_layers): ModuleList(
      (0-3): 4 x <Quantum Torch Layer: func=quantum_circuit>
    )
  )
  (fc2): Linear(in_features=20, out_features=10, bias=True)
)


In [18]:
x, label = next(iter(train_loader))

print("Input:")
print(x.shape)

x1 = model.cnn(x)

print("\nAfter CNN:")
print(x1.shape)

x2 = model.fc1(x1)

print("\nAfter classical dense:")
print(x2.shape)

import time

start = time.time()
x3 = model.quantum(x2)
end = time.time()

print("\nAfter quantum layer:")
print(x3.shape)
print(f"Quantum layer time: {end - start:.4f} seconds")

x4 = model.fc2(x3)

print("\nOutput:")
print(x4.shape)

Input:
torch.Size([32, 1, 28, 28])

After CNN:
torch.Size([32, 1568])

After classical dense:
torch.Size([32, 20])

After quantum layer:
torch.Size([32, 20])
Quantum layer time: 3.4123 seconds

Output:
torch.Size([32, 10])


In [19]:
def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


print(
    "Trainable parameters:",
    count_parameters(model)
)

Trainable parameters: 45154


In [20]:
for name, param in model.named_parameters():
    if "quantum" in name:
        print(
            name,
            param.shape,
            param.numel()
        )

quantum.quantum_layers.0.weights torch.Size([3, 5, 3]) 45
quantum.quantum_layers.1.weights torch.Size([3, 5, 3]) 45
quantum.quantum_layers.2.weights torch.Size([3, 5, 3]) 45
quantum.quantum_layers.3.weights torch.Size([3, 5, 3]) 45


In [21]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [22]:
# ============================================================
# DEVICE
# ============================================================

device = torch.device("cuda")

print("Using:", device)


# ============================================================
# CREATE MODEL
# ============================================================

model = HQNNParallel().to(device)


# ============================================================
# LOSS + OPTIMIZER
# ============================================================

criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

Using: cuda


In [23]:
print("PyTorch CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Model:", next(model.parameters()).device)
print("Quantum device:", qdev)

PyTorch CUDA: True
GPU: NVIDIA H100 PCIe
Model: cuda:0
Quantum device: <lightning.gpu device (wires=5) at 0x7fec9a9cb380>


In [24]:
from tqdm.auto import tqdm

epochs = 5
train_losses = []

for epoch in range(epochs):

    model.train()
    running_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{epochs}"
    )

    for images, labels in progress_bar:

        images = images.to(device)
        labels = labels.to(device)

        break
        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        # Show current batch loss in the progress bar
        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"- Mean loss: {epoch_loss:.4f}"
    )

Epoch 1/5:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 1/5 - Mean loss: 0.0000


Epoch 2/5:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 2/5 - Mean loss: 0.0000


Epoch 3/5:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 3/5 - Mean loss: 0.0000


Epoch 4/5:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 4/5 - Mean loss: 0.0000


Epoch 5/5:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 5/5 - Mean loss: 0.0000


In [25]:
model.eval()

with torch.no_grad():
    features = model.cnn(images.to(device))
    features = model.fc1(features)
    features = model.bn1(features)
    features = torch.relu(features)

print(features.shape)

torch.Size([32, 20])


In [26]:
features = features.cpu().numpy()

In [27]:
for name, param in model.named_parameters():
    if "quantum" in name:
        print(name, param.shape)

quantum.quantum_layers.0.weights torch.Size([3, 5, 3])
quantum.quantum_layers.1.weights torch.Size([3, 5, 3])
quantum.quantum_layers.2.weights torch.Size([3, 5, 3])
quantum.quantum_layers.3.weights torch.Size([3, 5, 3])


In [38]:
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator
)

service = QiskitRuntimeService()

backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=5
)

print(backend.name)

ibm_fez


In [41]:
model.eval()

images, labels = next(iter(test_loader))

# Just one image for the first hardware test
image = images[0:1].to(device)
label = labels[0].item()

with torch.no_grad():

    x = model.cnn(image)

    x = model.fc1(x)
    x = model.bn1(x)
    x = torch.relu(x)

features20 = x[0].detach().cpu().numpy()

print("True class:", label)
print("20 features:", features20)
print("Shape:", features20.shape)

True class: 7
20 features: [0.         0.07963645 0.04076171 0.         0.         0.03443102
 0.         0.01549811 0.         0.         0.02527543 0.00409815
 0.         0.         0.03358466 0.         0.12807788 0.02781298
 0.00850208 0.        ]
Shape: (20,)


In [42]:
for name, param in model.named_parameters():
    if "quantum" in name:
        print(name, param.shape)

quantum.quantum_layers.0.weights torch.Size([3, 5, 3])
quantum.quantum_layers.1.weights torch.Size([3, 5, 3])
quantum.quantum_layers.2.weights torch.Size([3, 5, 3])
quantum.quantum_layers.3.weights torch.Size([3, 5, 3])


In [43]:
quantum_weights = []

for layer in model.quantum.quantum_layers:

    w = layer.weights.detach().cpu().numpy()
    quantum_weights.append(w)

print(len(quantum_weights))
print(quantum_weights[0].shape)

4
(3, 5, 3)


In [44]:
import numpy as np

from qiskit import QuantumCircuit


def make_hqnn_qiskit_circuit(features, weights):
    """
    Recreate one trained 5-qubit HQNN PQC.

    features : shape (5,)
    weights  : shape (3, 5, 3)
    """

    n_qubits = 5

    qc = QuantumCircuit(n_qubits)

    # ---------------------------------------------------------
    # AngleEmbedding(rotation="X")
    # ---------------------------------------------------------

    for q in range(n_qubits):
        qc.rx(float(features[q]), q)

    # ---------------------------------------------------------
    # StronglyEntanglingLayers
    # ---------------------------------------------------------

    n_layers = weights.shape[0]

    for layer in range(n_layers):

        # General Rot(phi, theta, omega)
        for q in range(n_qubits):

            phi, theta, omega = weights[layer, q]

            # PennyLane Rot(phi, theta, omega)
            qc.rz(float(phi), q)
            qc.ry(float(theta), q)
            qc.rz(float(omega), q)

        # PennyLane default ranges are 1, 2, 3, ...
        r = (layer % (n_qubits - 1)) + 1

        # CNOT q -> q+r mod n_qubits
        for q in range(n_qubits):

            target = (q + r) % n_qubits

            qc.cx(q, target)

    return qc

In [45]:
from qiskit.quantum_info import SparsePauliOp

Y_observables = [
    SparsePauliOp("IIIIY"),  # Y on q0
    SparsePauliOp("IIIYI"),  # Y on q1
    SparsePauliOp("IIYII"),  # Y on q2
    SparsePauliOp("IYIII"),  # Y on q3
    SparsePauliOp("YIIII"),  # Y on q4
]

In [46]:
from qiskit.transpiler.preset_passmanagers import (
    generate_preset_pass_manager
)

from qiskit_ibm_runtime import EstimatorV2 as Estimator


pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=2
)

estimator = Estimator(
    mode=backend
)

# Explicitly control statistics
estimator.options.default_shots = 1000

In [47]:
def run_pqc_on_ibm(features5, weights, shots=1000):

    qc = make_hqnn_qiskit_circuit(
        features5,
        weights
    )

    # Convert to hardware-native ISA circuit
    isa_qc = pm.run(qc)

    # Observables must follow the transpiled layout
    isa_obs = [
        obs.apply_layout(isa_qc.layout)
        for obs in Y_observables
    ]

    # Control shots
    estimator.options.default_shots = shots

    job = estimator.run([
        (
            isa_qc,
            isa_obs
        )
    ])

    print("Job ID:", job.job_id())

    result = job.result()[0]

    mean = np.asarray(
        result.data.evs,
        dtype=float
    )

    std = np.asarray(
        result.data.stds,
        dtype=float
    )

    return mean, std

In [48]:
q_mean, q_std = run_pqc_on_ibm(
    features20[0:5],
    quantum_weights[0],
    shots=1000
)

Job ID: da53l061vhnc73flk9i0


In [49]:
for i, (mu, sig) in enumerate(
    zip(q_mean, q_std)
):
    print(
        f"<Y{i}> = "
        f"{mu:+.5f} ± {sig:.5f}"
    )

<Y0> = -0.09711 ± 0.04181
<Y1> = +0.12283 ± 0.03266
<Y2> = -0.30896 ± 0.03391
<Y3> = +0.18794 ± 0.03096
<Y4> = +0.23585 ± 0.02703


In [50]:
all_means = []
all_stds = []

for circuit_idx in range(4):

    start = circuit_idx * 5
    end = start + 5

    print(
        f"\nRunning PQC {circuit_idx + 1}/4"
    )

    mean, std = run_pqc_on_ibm(
        features20[start:end],
        quantum_weights[circuit_idx],
        shots=1000
    )

    print(
        f"<Y> = {mean} ± {std}"
    )
    all_means.append(mean)
    all_stds.append(std)


Running PQC 1/4
Job ID: da53ldk3jnrc73agraf0
<Y> = [-0.11105602  0.19467378 -0.28362774  0.23128243  0.1803431 ] ± [0.03306578 0.03213906 0.03118586 0.03758197 0.03736322]

Running PQC 2/4
Job ID: da53li6aa69c739kdhp0
<Y> = [ 0.2533556  -0.12080703 -0.20044654 -0.06181456 -0.16239634] ± [0.03755605 0.02674997 0.03159521 0.0337772  0.02255931]

Running PQC 3/4
Job ID: da53lmmaa69c739kdi10
<Y> = [-0.13490391 -0.1346368  -0.03576973 -0.23328556  0.04551076] ± [0.03325617 0.03423681 0.03394289 0.03321226 0.03637514]

Running PQC 4/4
Job ID: da53lr61vhnc73flkalg
<Y> = [-0.1547907  -0.0337448   0.16732026 -0.06520934 -0.08128369] ± [0.04373804 0.03718545 0.02447767 0.02687257 0.0320863 ]


In [51]:
qpu_mean = np.concatenate(all_means)
qpu_std = np.concatenate(all_stds)

print(qpu_mean.shape)
print(qpu_std.shape)

(20,)
(20,)


In [52]:
qpu_tensor = torch.tensor(
    qpu_mean,
    dtype=torch.float32,
    device=device
).unsqueeze(0)

with torch.no_grad():

    logits_qpu = model.fc2(
        qpu_tensor
    )

    probs_qpu = torch.softmax(
        logits_qpu,
        dim=1
    )

pred_qpu = probs_qpu.argmax(dim=1).item()

print("True class :", label)
print("QPU prediction:", pred_qpu)
print("Probabilities:", probs_qpu.cpu().numpy())

True class : 7
QPU prediction: 3
Probabilities: [[0.08556805 0.1122032  0.1090746  0.11872816 0.10492385 0.07886071
  0.08584993 0.0831352  0.10365967 0.11799666]]


In [53]:
N_MC = 10000

mu = torch.tensor(
    qpu_mean,
    dtype=torch.float32,
    device=device
)

sigma = torch.tensor(
    qpu_std,
    dtype=torch.float32,
    device=device
)

# Draw 10,000 possible quantum-layer realizations
q_samples = (
    mu[None, :]
    +
    torch.randn(
        N_MC,
        20,
        device=device
    )
    * sigma[None, :]
)

with torch.no_grad():

    logits_samples = model.fc2(
        q_samples
    )

    probs_samples = torch.softmax(
        logits_samples,
        dim=1
    )

In [55]:
prob_mean = (
    probs_samples
    .mean(dim=0)
    .cpu()
    .numpy()
)

prob_std = (
    probs_samples
    .std(dim=0)
    .cpu()
    .numpy()
)

for c in range(10):

    print(
        f"Class {c}: "
        f"{prob_mean[c]:.4f} "
        f"± {prob_std[c]:.4f}"
    )

Class 0: 0.0856 ± 0.0018
Class 1: 0.1122 ± 0.0021
Class 2: 0.1091 ± 0.0020
Class 3: 0.1187 ± 0.0024
Class 4: 0.1049 ± 0.0021
Class 5: 0.0789 ± 0.0014
Class 6: 0.0859 ± 0.0015
Class 7: 0.0831 ± 0.0017
Class 8: 0.1037 ± 0.0017
Class 9: 0.1179 ± 0.0023


In [56]:
target_mc = torch.full(
    (N_MC,),
    label,
    dtype=torch.long,
    device=device
)

losses_mc = torch.nn.functional.cross_entropy(
    logits_samples,
    target_mc,
    reduction="none"
)

print(
    "Loss = "
    f"{losses_mc.mean().item():.5f} "
    f"± {losses_mc.std().item():.5f}"
)

Loss = 2.48731 ± 0.01997
